# Spotify Playlist Clustering and GRU Recommendation

**ML class final project, May 2025**  
Report authors: Tony Lai and Ian Jang.

This notebook organizes the code segments preserved in the [original full report](reports/Spotify_ML_Project_Report.pdf). It is a **report-based archive, not the recovered original training notebook**. All code cells are unexecuted and all result tables below are transcribed historical results, not generated outputs.

The report includes playlist aggregation, PCA, a MiniBatchKMeans search (with the old KMeans version commented out), and one GRU prediction call. It does not include dataset loading/join code, the full neural-network implementation, negative sampling, train/test construction, Optuna objectives, or GPU training code. Those missing pieces have not been invented.

No dataset, model weights, credentials, CUDA setup, or trained model is included. Do not expect **Run All** to work without restoring the missing upstream artifacts.

## 1. Data sources and prerequisites

- **Spotify Million Playlist Dataset:** existing user-created playlists, track identifiers, and track positions.
- **SQLite audio-feature database:** a lookup table described in the report as approximately two million songs with audio descriptors. The report does not give an exact download URL, schema, or license; these must be recovered before reproducing the join.

The snippets assume an existing pandas DataFrame named `full_df` with one row per playlist-track occurrence, a `pid` column, and the nine numeric columns below. The report describes joining the sources by Spotify track ID but does not show the join implementation or missing-track policy.

`numpy` and `pandas` imports below are packaging additions. The model code shown here uses CPU scikit-learn; the report discusses GPU experiments but does not preserve their implementation.

In [ ]:
import numpy as np
import pandas as pd


## 2. Summarize each playlist

**Source: report section 2.3, 'Merging the two datasets together'.**

Aggregate the existing song features by playlist ID, retaining each feature's mean and standard deviation. This produces **18 features** (nine means plus nine standard deviations), not 36. The report's incorrect shape comment is corrected below; the computation is unchanged.

Standard deviations represent within-playlist variability. No special mixed-playlist splitting or outlier-removal procedure is documented. `fillna(0)` replaces missing aggregate values; it is not evidence of a complete missing-data cleaning policy.

In [ ]:
num_cols = ['danceability','energy','loudness','speechiness',
            'acousticness','instrumentalness','liveness',
            'valence','tempo']
pl_vec = (
    full_df.groupby('pid')[num_cols]
           .agg(['mean','std'])
           .fillna(0)
)
pl_vec.columns = ['_'.join(c) for c in pl_vec.columns]  # flatten
print(pl_vec.shape)  # (number of playlists, 18)


## 3. PCA before clustering

**Source: report section 3.1, 'MiniBatchKMeans'.**

Retain enough principal components to explain 95% of input variance. This is a variance-retention setting, not a claim of 95% predictive accuracy.

**Reproducibility note:** the preserved snippet has no explicit standardization step. Earlier preprocessing may have existed in the original notebook, but it cannot be verified from this report. Do not claim standardization was definitely performed or definitely omitted. The separate normalized-centroid figures in the report are visualization scaling, not proof of input standardization.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95, random_state=42)
X_red = pca.fit_transform(pl_vec.values)


## 4. MiniBatchKMeans search

**Source: report section 3.1.**

Search `k=4..20` with batches of 2,048. Compute silhouette and Davies-Bouldin on a fixed 10,000-playlist subset, then select the model with the highest silhouette score. Davies-Bouldin is reported but is not the selection criterion in this snippet.

**Preconditions and limits:**
- This original snippet requires at least 10,000 rows because sampling is without replacement. For smaller datasets it must be adapted; the report's 1,000-playlist results cannot be reproduced by this snippet unchanged.
- `.fit(X_red)` still receives the full feature matrix; this is not an out-of-core `partial_fit` implementation.
- The original `n_init='auto'` setting is retained, but the report's misleading comment equating it with parallel execution is removed.
- The old full KMeans search is retained as comments, as it was in the report. Its evaluation also used the full dataset, so the reported speed change cannot be attributed solely to mini-batching.

In [ ]:
# Fast grid-search with MiniBatchKMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Take a 10-k playlist sample for metrics -- reproducible with seed
rng      = np.random.default_rng(42)
idx_samp = rng.choice(len(X_red), size=10_000, replace=False)
X_samp   = X_red[idx_samp]

best = None
for k in range(4, 21):
    km = MiniBatchKMeans(
            n_clusters=k,
            batch_size=2048,
            max_iter=100,
            n_init='auto',
            random_state=42,
            reassignment_ratio=0.01
    ).fit(X_red)

    # Evaluate clustering quality on the 10-k sample
    labels_samp = km.labels_[idx_samp]
    sil = silhouette_score(X_samp, labels_samp)
    db  = davies_bouldin_score(X_samp, labels_samp)

    print(f"k={k:2d}  sil={sil:.4f}  DB={db:.4f}")

    if best is None or sil > best[1]:
        best = (km, sil, db)

# Attach winning labels to the full playlist index
pl_vec["cluster"] = best[0].labels_

# Original deprecated full KMeans search, retained as comments:
# from sklearn.cluster import KMeans
# from sklearn.metrics import silhouette_score, davies_bouldin_score
# best = None
# for k in range(4,21):
#     km = KMeans(k, n_init='auto', random_state=42).fit(X_red)
#     sil = silhouette_score(X_red, km.labels_)
#     db  = davies_bouldin_score(X_red, km.labels_)
#     print(f"k={k:2d}  sil={sil:.3f}  DB={db:.3f}")
#     if best is None or sil > best[1]:
#         best = (km, sil, db)
# pl_vec['cluster'] = best[0].labels_


### Reported clustering results

Transcribed from report Table 1; **not rerun**. The GPU implementation and Optuna search are not present in the available code excerpts.

| Metric | Baseline MiniBatchKMeans | Report's GPU + Optuna version |
|---|---:|---:|
| Silhouette (higher is better) | 0.2821 | 0.2939 |
| Davies-Bouldin (lower is better) | 1.0424 | 1.0220 |
| Selected k | 4 | 4 |

These are small improvements in cluster geometry, not recommendation accuracy. No repeated-run uncertainty estimates or isolated GPU-vs-tuning ablation are available.

## 5. How clustering connects to the GRU

**Documented design, not recovered sampler code:**

1. Keep the original playlist order as the supervision signal.
2. Given a partial playlist, label the observed next song as positive.
3. Sample alternative tracks from the same playlist cluster as negatives.
4. Train a scorer to distinguish the observed continuation from those alternatives.

Illustrative example: original playlist `A -> B -> C -> D`; context `A, B, C`; positive candidate `D`; negative candidate `X` sampled from the same cluster's track pool.

Cluster assignments are **not ground-truth labels**. A sampled negative is not necessarily a bad recommendation. The report does not specify sampling fallback rules, duplicate/positive exclusions, or every slicing detail. It also does not establish a complete inference-time restriction to same-cluster candidates. No sampler is implemented here.

## 6. GRU candidate scorer

**Source: report sections 3.3-3.4.** The report describes:

- Ordered track sequences with nine audio features per track.
- A sequence length of 100, with shorter sequences padded and masked.
- A GRU producing a 64-dimensional sequence representation.
- Concatenation with a candidate track's nine features to obtain 73 values.
- A feedforward network producing a score for the context-candidate pair.

Only the prediction call below is preserved. `model`, `X_seq_te`, and `X_cand_te` must come from the missing original model/data preparation code. This notebook does not construct, train, download, or load a model. The report describes an 80/20 split, but the code needed to verify playlist-level split isolation is absent.

In [ ]:
proba_te = model.predict({"seq_in": X_seq_te, "cand_in": X_cand_te})


## 7. Historical GRU results

Transcribed from report section 3.4 and Table 2; **not new notebook outputs**.

| Playlists | Baseline ROC-AUC | Baseline reported MAP | Optuna ROC-AUC | Optuna reported MAP |
|---|---:|---:|---:|---:|
| 1,000 | 0.515 | 0.346 | 0.460 | 0.306 |
| 10,000 | 0.540 | 0.351 | 0.614 | 0.407 |
| 30,000 | 0.672 | 0.470 | 0.693 | 0.495 |
| 50,000 | 0.708 | 0.507 | 0.708 | 0.507 |

- ROC-AUC measures discrimination between observed-next-track positives and sampled negatives. A value of 0.708 is not 70.8% next-song prediction accuracy across the full catalog.
- The report calls its second metric MAP but also describes binary average precision. Without the metric implementation, pooled AP versus per-playlist MAP remains unresolved.
- Tuning did not improve every setting: results worsened at 1,000 playlists and were unchanged at 50,000.
- The report records RAM/time limits and unstable runs at 100,000+ playlists, with no completed larger-scale metrics.
- There is no controlled comparison showing that clustering-based negatives or sequence order independently improved recommendation quality.

## 8. What is needed for a runnable reproduction

Recover the original data loader and SQLite schema, feature-join/missing-data policy, preprocessing/scaling, train/test split, negative sampler, sequence construction, GRU definition/training, metric implementation, and Optuna/GPU configuration. Use a held-out evaluation protocol and compare the same model with random versus cluster-informed negatives on identical test candidates.

The unchanged PDF remains the historical source, including its original wording and inconsistencies. This notebook's annotations distinguish preserved code from explanatory additions. No cells were executed while preparing this archive.